# Structured data classification with Keras

Classificacao binaria de doenca cardiaca com features numericas e categoricas.

O notebook reproduz a implementacao do arquivo `structured_data_classification_keras.py`.

In [1]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
import pandas as pd
import tensorflow as tf
from keras import layers

keras.utils.set_random_seed(1337)

TARGET_FEATURE_NAME = "target"
NUMERIC_FEATURE_NAMES = ["age", "trestbps", "thalach", "oldpeak", "slope", "chol"]
CATEGORICAL_FEATURE_NAMES = ["sex", "cp", "fbs", "restecg", "exang", "ca", "thal"]
FEATURE_NAMES = NUMERIC_FEATURE_NAMES + CATEGORICAL_FEATURE_NAMES

dataframe = pd.read_csv("http://storage.googleapis.com/download.tensorflow.org/data/heart.csv")
val_dataframe = dataframe.sample(frac=0.2, random_state=1337)
train_dataframe = dataframe.drop(val_dataframe.index)
print(dataframe.shape, len(train_dataframe), len(val_dataframe))

(303, 14) 242 61


## Pre-processamento

As estatisticas de normalizacao e os vocabularios sao aprendidos somente com o conjunto de treino.

In [2]:
categorical_is_string = {
    name: not pd.api.types.is_numeric_dtype(train_dataframe[name])
    for name in CATEGORICAL_FEATURE_NAMES
}

categorical_encoders = {}
for name in CATEGORICAL_FEATURE_NAMES:
    values = train_dataframe[name]
    if categorical_is_string[name]:
        encoder = layers.StringLookup(
            vocabulary=sorted(values.astype(str).unique()),
            mask_token=None, num_oov_indices=1, output_mode="one_hot"
        )
    else:
        encoder = layers.IntegerLookup(
            vocabulary=sorted(values.astype("int64").unique()),
            mask_token=None, num_oov_indices=1, output_mode="one_hot"
        )
    categorical_encoders[name] = encoder

normalizers = {}
for name in NUMERIC_FEATURE_NAMES:
    normalizer = layers.Normalization(axis=-1)
    normalizer.adapt(train_dataframe[name].to_numpy(dtype="float32").reshape(-1, 1))
    normalizers[name] = normalizer

def dataframe_to_dataset(frame, shuffle=False):
    frame = frame.copy()
    labels = frame.pop(TARGET_FEATURE_NAME).to_numpy(dtype="float32")
    features = {}
    for name in FEATURE_NAMES:
        features[name] = frame[name].astype(str).to_numpy() if categorical_is_string.get(name, False) else frame[name].to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((features, labels))

    def encode(features, label):
        encoded = {}
        for name in NUMERIC_FEATURE_NAMES:
            encoded[name] = tf.expand_dims(tf.cast(features[name], tf.float32), -1)
        for name in CATEGORICAL_FEATURE_NAMES:
            value = tf.cast(features[name], tf.string if categorical_is_string[name] else tf.int64)
            encoded[name] = tf.reshape(categorical_encoders[name](value), [-1])
        return encoded, label

    dataset = dataset.map(encode)
    if shuffle:
        dataset = dataset.shuffle(len(frame), seed=1337)
    return dataset.batch(32).prefetch(tf.data.AUTOTUNE)

train_ds = dataframe_to_dataset(train_dataframe, shuffle=True)
val_ds = dataframe_to_dataset(val_dataframe)

## Modelo e treinamento

In [3]:
model_inputs = {}
processed_features = []

for name in NUMERIC_FEATURE_NAMES:
    feature_input = layers.Input(name=name, shape=(1,), dtype=tf.float32)
    model_inputs[name] = feature_input
    processed_features.append(normalizers[name](feature_input))

for name in CATEGORICAL_FEATURE_NAMES:
    feature_input = layers.Input(name=name, shape=(categorical_encoders[name].vocabulary_size(),), dtype=tf.float32)
    model_inputs[name] = feature_input
    processed_features.append(feature_input)

features = layers.Concatenate()(processed_features)
features = layers.Dense(32, activation="relu")(features)
features = layers.Dropout(0.5)(features)
output = layers.Dense(1, activation="sigmoid")(features)
model = keras.Model(model_inputs, output)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[keras.metrics.BinaryAccuracy(name="accuracy"), keras.metrics.AUC(name="auc")],
)
model.summary()

history = model.fit(
    train_ds, validation_data=val_ds, epochs=30, shuffle=False, verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True)],
)

metrics = model.evaluate(val_ds, return_dict=True, verbose=0)
print({name: round(float(value), 4) for name, value in metrics.items()})

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ age (InputLayer)    │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ trestbps            │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ thalach             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ oldpeak             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ slope (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ chol (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 1)         │          3 │ age[0][0]         │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 1)         │          3 │ trestbps[0][0]    │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_2     │ (None, 1)         │          3 │ thalach[0][0]     │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_3     │ (None, 1)         │          3 │ oldpeak[0][0]     │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_4     │ (None, 1)         │          3 │ slope[0][0]       │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_5     │ (None, 1)         │          3 │ chol[0][0]        │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sex (InputLayer)    │ (None, 3)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cp (InputLayer)     │ (None, 6)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fbs (InputLayer)    │ (None, 3)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ restecg             │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ exang (InputLayer)  │ (None, 3)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ca (InputLayer)     │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ thal (InputLayer)   │ (None, 6)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 36)        │          0 │ normalization[0]

 Total params: 1,235 (4.85 KB)

 Trainable params: 1,217 (4.75 KB)

 Non-trainable params: 18 (96.00 B)

{'accuracy': 0.7869, 'auc': 0.8617, 'loss': 0.3835}


## Inferencia

In [4]:
sample = {"age": 60, "sex": 1, "cp": 1, "trestbps": 145, "chol": 233, "fbs": 1, "restecg": 2, "thalach": 150, "exang": 0, "oldpeak": 2.3, "slope": 3, "ca": 0, "thal": "fixed"}
sample_inputs = {}
for name in NUMERIC_FEATURE_NAMES:
    sample_inputs[name] = tf.constant([[sample[name]]], dtype=tf.float32)
for name in CATEGORICAL_FEATURE_NAMES:
    value = tf.constant([str(sample[name])]) if categorical_is_string[name] else tf.constant([sample[name]], dtype=tf.int64)
    sample_inputs[name] = categorical_encoders[name](value)
prediction = float(model.predict(sample_inputs, verbose=0)[0, 0])
print(f"Probabilidade estimada: {prediction:.1%}")
print(f"Classe prevista: {int(prediction >= 0.5)}")

Probabilidade estimada: 59.4%
Classe prevista: 1
